# 06 - Defining Response Curves

**Inputs:**
- `../data/processed/storm_events.pkl` — county-month storm observations
- `../data/processed/zillow_panel.pkl` — affected county-month ZHVI vs regional baseline
- `../data/processed/nri_panel.pkl` — NRI scores by county-year
- Zillow raw CSV — for pre-storm ZHVI lookups


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import urllib.request

Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

## Load Processed Data

In [2]:
storms   = pd.read_pickle('../data/processed/storm_events.pkl')
zillow   = pd.read_pickle('../data/processed/zillow_panel.pkl')
nri      = pd.read_pickle('../data/processed/nri_panel.pkl')

print(f'Storm events:  {storms.shape}')
print(f'Zillow panel:  {zillow.shape}')
print(f'NRI panel:     {nri.shape}')

Storm events:  (14660, 7)
Zillow panel:  (14166, 10)
NRI panel:     (18870, 12)


## EDA

In [3]:
df = pd.read_csv('../data/processed/analysis_dataset.csv')
df.head()

storms['month_index'] = (storms['year']-2020)*12 + storms['month']

In [6]:
county_storm_counts = storms.sort_values(['stcofips', 'month_index'],ascending=True).groupby(['stcofips', 'month_index']).size().reset_index(name='storm_count')
county_storm_counts['months_next_storm'] = -county_storm_counts.groupby(['stcofips'])['month_index'].diff(-1)
county_storm_counts['months_last_storm'] = county_storm_counts.groupby(['stcofips'])['month_index'].diff(1)

for window_size in range(1,13):
    isolated_storm_events = county_storm_counts[(county_storm_counts['months_next_storm'] > 36) & (county_storm_counts['months_last_storm'] > 12)]
    print(f"{window_size}: {len(isolated_storm_events)}")

1: 22
2: 22
3: 22
4: 22
5: 22
6: 22
7: 22
8: 22
9: 22
10: 22
11: 22
12: 22


In [5]:
storms

,stcofips,year,month,event_count,total_damage,log_damage,total_duration_days,month_index
0,01001,2021,4,2,0.0,0.000000,0.001389,16
1,01001,2021,5,1,0.0,0.000000,0.001389,17
2,01001,2025,4,1,0.0,0.000000,0.003472,64
3,01001,2025,5,1,0.0,0.000000,0.000694,65
4,01001,2025,7,1,0.0,0.000000,0.003472,67
...,...,...,...,...,...,...,...,...
14655,99083,2024,9,1,100.0,4.615121,0.013889,57
14656,99097,2024,8,2,1000.0,6.908755,0.000000,56
14657,99131,2021,9,2,1000.0,6.908755,0.034722,21
14658,99135,2021,9,2,1000.0,6.908755,0.000000,21
